<div align="center">

# Jackrong-llm-finetuning-guide 🌌
**An Educational, End-to-End LLM Fine-Tuning Pipeline for Beginners and Developers**

Fine-tune **Qwopus3.8-27B in Google Colab** with Unsloth, 4-bit QLoRA, and public reasoning examples.

🌐 **Language:**  🇬🇧 **English** 

🤗 **HuggingFace:** [Jackrong](https://huggingface.co/Jackrong)

<br>

[![Unsloth](https://img.shields.io/badge/Powered%20by-Unsloth-8A2BE2?style=flat-square)](https://github.com/unslothai/unsloth)
[![Google Colab](https://img.shields.io/badge/Environment-Google%20Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/Framework-PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Hugging Face](https://img.shields.io/badge/Model%20Hub-Hugging%20Face-FFD21E?style=flat-square&logo=huggingface&logoColor=black)](https://huggingface.co/)
[![LoRA PEFT](https://img.shields.io/badge/Technique-LoRA%20%2F%20PEFT-007EC6?style=flat-square)](#)
[![Beginner Friendly](https://img.shields.io/badge/Level-Beginner%20Friendly-brightgreen?style=flat-square)](#)

</div>

---

## Before You Start 🔑

Select an A100-class GPU in Google Colab. Add `WANDB_API_KEY` in Colab Secrets for training logs and `HF_TOKEN` for the optional Hugging Face uploads. Turn on notebook access for each secret you use.

Training checkpoints go to Google Drive. Run the final two cells only for the exports you want to upload.

In [ ]:
%%capture
import os

# Adapted from the Unsloth Qwen3.8-27B Colab notebook.
# https://github.com/unslothai/notebooks/blob/main/nb/Qwen3.8_%2827B%29-Conversational.ipynb
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==5.15.1 wandb
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen3.8-27B",
    max_seq_length = 32768,  # Set the maximum training context for your workload and GPU memory.
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    device_map = "cuda:0",  # Place the full model on GPU 0.
)
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,  # This dataset is text-only.
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 64,  # A higher LoRA rank adds capacity and uses more VRAM.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "out_proj",
    ],
    lora_alpha = 64,  # Scales the strength of LoRA updates.
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## Dataset Processing Overview 📚

The Qwopus3.8-27B training mix was assembled locally from many datasets. This notebook uses two public reasoning datasets to demonstrate live preprocessing in Colab. You can substitute your own SFT datasets to train for other tasks or domains.

1. **Load and sample.** Take 1,000 rows each from the `Math` and `STEM` subsets of [DeepSeek-V4-Pro-Distilled-200K](https://huggingface.co/datasets/Jackrong/DeepSeek-V4-Pro-Distilled-200K), and 1,000 from [Claude-opus-4.7-TraceInversion-5000x](https://huggingface.co/datasets/Jackrong/Claude-opus-4.7-TraceInversion-5000x).
2. **Separate reasoning and answers.** DeepSeek stores both in `output`. TraceInversion stores the reconstructed reasoning in `inverted_reasoning` and the answer in `output`.
3. **Apply the native chat template.** Pass the trace as `reasoning_content` and the answer as `content`. The Qwen3.8 tokenizer renders the assistant turn as shown below. Set `reasoning_effort` to `low`, `medium`, or `xhigh`.
4. **Mix, shuffle, and filter.** Remove incomplete rows and keep formatted examples up to 8,192 tokens.

```text
<think>
reasoning
</think>

final answer
```


In [ ]:
from datasets import Dataset, concatenate_datasets, load_dataset

# Edit the datasets, sample counts, and token limit here.
deepseek_dataset = "Jackrong/DeepSeek-V4-Pro-Distilled-200K"
deepseek_samples = {"Math": 1000, "STEM": 1000}
claude_dataset = "Jackrong/Claude-opus-4.7-TraceInversion-5000x"
claude_subset = "default"
claude_samples = 1000
max_context_window = 8192  # Tokens per formatted conversation.
reasoning_effort = "medium"
seed = 12181531

def load_and_sample(dataset_name, subset, sample_count, format_row):
    rows = load_dataset(dataset_name, subset, split="train", streaming=True)
    rows = rows.shuffle(seed=seed, buffer_size=128).take(sample_count)
    return Dataset.from_list([format_row(row) for row in rows])

def format_deepseek(row):
    output = (row["output"] or "").strip().removeprefix("<think>")
    reasoning, _, answer = output.partition("</think>")
    return {
        "prompt": (row["input"] or "").strip(),
        "reasoning": reasoning.strip(),
        "answer": answer.strip(),
    }

def format_claude(row):
    reasoning = (row["inverted_reasoning"] or "").strip()
    reasoning = reasoning.removeprefix("<think>").removesuffix("</think>")
    return {
        "prompt": (row["input"] or "").strip(),
        "reasoning": reasoning.strip(),
        "answer": (row["output"] or "").strip(),
    }

dataset_parts = [
    load_and_sample(deepseek_dataset, subset, sample_count, format_deepseek)
    for subset, sample_count in deepseek_samples.items()
]
dataset_parts.append(load_and_sample(claude_dataset, claude_subset, claude_samples, format_claude))

combined_dataset = concatenate_datasets(dataset_parts).shuffle(seed=seed)
combined_dataset = combined_dataset.filter(
    lambda row: all((row["prompt"], row["reasoning"], row["answer"]))
)

def format_chat(row):
    conversation = [
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "reasoning_content": row["reasoning"], "content": row["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False,
        reasoning_effort=reasoning_effort,
    )
    return {"text": text.rstrip("\n")}

dataset = combined_dataset.map(format_chat, remove_columns=combined_dataset.column_names)

def filter_long_sequences(examples):
    input_ids = text_tokenizer(
        examples["text"], truncation=False, add_special_tokens=False, return_attention_mask=False,
    )["input_ids"]
    return [len(tokens) <= max_context_window for tokens in input_ids]

dataset = dataset.filter(filter_long_sequences, batched=True, batch_size=16)
print(dataset[0]["text"][:800])

In [ ]:
import wandb
from google.colab import drive, userdata
from trl import SFTTrainer, SFTConfig

drive.mount("/content/drive")
wandb.login(key=userdata.get("WANDB_API_KEY"))

trainer = SFTTrainer(
    model = model,
    processing_class = text_tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_context_window,
        per_device_train_batch_size = 6,  # Reduce to 1 on a smaller GPU.
        gradient_accumulation_steps = 6,
        warmup_steps = 0.05,  # Warm up over the first 5% of training steps.
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        save_steps = 200,  # Save a checkpoint every 200 optimizer steps.
        save_total_limit = 1,  # Keep only the newest checkpoint to limit Drive usage.
        save_strategy = "steps",
        report_to = "wandb",
        output_dir = "/content/drive/MyDrive/Qwopus3.8-27B-checkpoints",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n<think>",
)

In [ ]:
text_tokenizer.decode(trainer.train_dataset[0]["input_ids"])

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("Qwopus3.8-27B-LoRA")
tokenizer.save_pretrained("Qwopus3.8-27B-LoRA")

In [ ]:
# Optional: upload merged 16-bit weights to Hugging Face.
from google.colab import userdata
from huggingface_hub import whoami

hf_token = userdata.get("HF_TOKEN")
username = whoami(token=hf_token)["name"]
repo_id = f"{username}/Qwopus3.8-27B-merged-16bit"

model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

In [ ]:
# Optional: upload GGUF files to Hugging Face.
from google.colab import userdata
from huggingface_hub import whoami

hf_token = userdata.get("HF_TOKEN")
username = whoami(token=hf_token)["name"]
repo_id = f"{username}/Qwopus3.8-27B-GGUF"

model.push_to_hub_gguf(
    repo_id,
    tokenizer,
    quantization_method=["q4_k_m", "q8_0", "bf16"],
    token=hf_token,
)